# Projet Algorithmique Python - Sujet 2
## Partie 1 : Web Scraping - France Info
**Master 1 IdL - Semestre 2 - 2025/2026**

---

### Objectif
Ce notebook collecte automatiquement des articles de presse depuis le site **France Info** et les sauvegarde dans un fichier CSV avec leurs categories (labels).

Ces donnees seront ensuite utilisees dans la **Partie 2** pour entrainer un modele Transformer.

## Etape 1 : Installation des dependances

On installe les bibliotheques necessaires :
- **requests** : pour envoyer des requetes HTTP et telecharger les pages web
- **lxml** : pour analyser le HTML et extraire le texte des articles

In [ ]:
# Installation des bibliotheques
# A executer une seule fois
import sys
!{sys.executable} -m pip install requests lxml

## Etape 2 : Importation des bibliotheques

On importe toutes les bibliotheques dont on a besoin :
- **requests** : envoie les requetes vers les sites web
- **lxml.html** : analyse le HTML et extrait les donnees
- **csv** : sauvegarde les donnees dans un fichier CSV
- **time** : fait des pauses entre les requetes
- **os** : gere les fichiers sur le disque

In [ ]:
import requests
from lxml import html
import csv
import time
import os

print("Bibliotheques importees avec succes")

## Etape 3 : Configuration

On definit les parametres du scraping :
- Les **sections** a scraper avec leurs labels (categories)
- Le **nombre d'articles** a recuperer par categorie
- Le **nom du fichier CSV** de sortie
- Le **User-Agent** pour simuler un vrai navigateur

In [ ]:
# Sections du site France Info avec leurs labels
# Chaque cle = le label, chaque valeur = l'URL de la section
SECTIONS = {
    "politique": "https://www.francetvinfo.fr/politique/",
    "sport":     "https://www.francetvinfo.fr/sports/",
    "economie":  "https://www.francetvinfo.fr/economie/",
    "sante":     "https://www.francetvinfo.fr/sante/",
    "sciences":  "https://www.francetvinfo.fr/sciences/",
}

# Nombre d'articles a recuperer par categorie
NB_ARTICLES = 10

# Nom du fichier CSV de sortie
FICHIER_CSV = "articles.csv"

# Simule un vrai navigateur pour ne pas etre bloque par le site
HEADERS = {"User-Agent": "Mozilla/5.0"}

print("Configuration OK")
print("Sections a scraper : " + str(list(SECTIONS.keys())))

## Etape 4 : Fonction de scraping

La fonction `scraper_section` fait le travail principal :
1. Elle telecharge la page de la section
2. Elle recupere tous les liens de la page
3. Elle visite chaque article
4. Elle extrait le texte de l'article
5. Elle retourne une liste d'articles avec leurs labels

In [ ]:
def scraper_section(url_section, label, nb_articles=10):
    """
    Recupere les articles d'une section France Info.
    Entree : url_section (str), label (str), nb_articles (int)
    Sortie : liste de dicts {url, texte, label}
    """
    resultats = []

    # Telecharger la page de la section
    try:
        reponse = requests.get(url_section, headers=HEADERS, timeout=10)
        reponse.raise_for_status()
    except requests.exceptions.Timeout:
        print("Erreur : timeout sur la section " + label)
        return resultats
    except requests.exceptions.HTTPError as e:
        print("Erreur HTTP sur " + label + " : " + str(e))
        return resultats
    except requests.exceptions.RequestException as e:
        print("Erreur reseau sur " + label + " : " + str(e))
        return resultats

    # Analyser le HTML avec lxml
    tree = html.fromstring(reponse.content)

    # Recuperer tous les liens de la page
    tous_les_liens = tree.xpath("//a/@href")

    # Eviter les doublons
    urls_visitees = set()
    compteur = 0

    for href in tous_les_liens:

        if compteur >= nb_articles:
            break

        if not href:
            continue

        # Construire l'URL complete si le lien est relatif
        if href.startswith("/"):
            href = "https://www.francetvinfo.fr" + href

        # Garder seulement les liens France Info
        if "francetvinfo.fr" not in href:
            continue

        # Ignorer les liens non-articles
        chemins_interdits = [
            ".jpg", ".png", ".mp4", ".gif",
            "/recherche/", "/profile/",
            "/mail/", "/user/", "/auth/",
            "/live/", "/en-direct/",
            "?", "#",
            "/tags/", "/auteur/",
            "/videos/", "/emissions/",
        ]
        if any(chemin in href for chemin in chemins_interdits):
            continue

        # Garder seulement les articles (finissent par .html)
        if not href.endswith(".html"):
            continue

        # Ignorer les doublons
        if href in urls_visitees:
            continue
        urls_visitees.add(href)

        # Visiter l'article
        try:
            reponse_article = requests.get(href, headers=HEADERS, timeout=10)
            reponse_article.raise_for_status()
        except:
            print("Erreur sur : " + href)
            continue

        # Extraire le texte avec lxml
        tree_article = html.fromstring(reponse_article.content)
        paragraphes = tree_article.xpath("//p//text()")

        if paragraphes:
            texte_brut = "\n".join(paragraphes)
        else:
            texte_brut = " ".join(tree_article.xpath("//text()"))

        # Nettoyer le texte
        lignes_propres = []
        for ligne in texte_brut.splitlines():
            ligne_nettoyee = ligne.strip()
            if ligne_nettoyee:
                lignes_propres.append(ligne_nettoyee)

        texte_final = "\n".join(lignes_propres)

        # Ignorer les articles trop courts
        if len(texte_final) < 300:
            print("Article trop court ignore : " + href)
            continue

        # Ajouter l'article aux resultats
        resultats.append({
            "url":   href,
            "texte": texte_final,
            "label": label
        })

        print("Article " + str(compteur + 1) + "/" + str(nb_articles) + " [" + label + "] : " + href)
        compteur += 1
        time.sleep(1)

    return resultats

print("Fonction scraper_section definie")

## Etape 5 : Fonction de sauvegarde CSV

La fonction `sauvegarder_csv` sauvegarde les articles dans un fichier CSV.

Le fichier CSV aura 3 colonnes :
- **url** : l'adresse de l'article
- **texte** : le contenu de l'article
- **label** : la categorie de l'article

In [ ]:
def sauvegarder_csv(donnees, nom_fichier):
    """
    Sauvegarde les articles dans un fichier CSV.
    Entree : donnees (list), nom_fichier (str)
    Sortie : fichier CSV sur le disque
    """
    fichier_existe = os.path.exists(nom_fichier)

    with open(nom_fichier, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["url", "texte", "label"])

        # Ecrire l'en-tete seulement si le fichier est nouveau
        if not fichier_existe:
            writer.writeheader()

        for article in donnees:
            writer.writerow(article)

    print(str(len(donnees)) + " articles sauvegardes dans " + nom_fichier)

print("Fonction sauvegarder_csv definie")

## Etape 6 : Fonction de statistiques

La fonction `afficher_statistiques` analyse le fichier CSV et affiche :
- Le nombre total d'articles
- La longueur moyenne des textes
- La repartition par label

In [ ]:
def afficher_statistiques(nom_fichier):
    """
    Affiche les statistiques du dataset.
    Entree : nom_fichier (str)
    Sortie : affichage dans le terminal
    """
    if not os.path.exists(nom_fichier):
        print("Fichier CSV introuvable.")
        return

    stats = {}
    longueurs = []

    with open(nom_fichier, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for ligne in reader:
            label = ligne["label"]
            texte = ligne["texte"]

            if label not in stats:
                stats[label] = 0
            stats[label] += 1
            longueurs.append(len(texte))

    total = sum(stats.values())
    longueur_moyenne = sum(longueurs) / len(longueurs) if longueurs else 0

    print("--- STATISTIQUES DU DATASET ---")
    print("Total articles   : " + str(total))
    print("Longueur moyenne : " + str(int(longueur_moyenne)) + " caracteres")
    print("")
    print("Repartition par label :")
    for label, count in stats.items():
        print("  " + label + " : " + str(count) + " articles")
    print("-------------------------------")

print("Fonction afficher_statistiques definie")

## Etape 7 : Lancement du scraping

On lance maintenant le scraping pour toutes les sections.

**Attention** : cette cellule peut prendre quelques minutes car on visite de nombreuses pages web avec une pause d'1 seconde entre chaque requete.

In [ ]:
# Supprimer l'ancien fichier CSV pour repartir a zero
if os.path.exists(FICHIER_CSV):
    os.remove(FICHIER_CSV)
    print("Ancien fichier supprime.")

# Scraper chaque section une par une
for label, url in SECTIONS.items():
    print("")
    print("Section : " + label)
    print("URL : " + url)

    articles = scraper_section(url, label, NB_ARTICLES)

    if articles:
        sauvegarder_csv(articles, FICHIER_CSV)
    else:
        print("Aucun article recupere pour : " + label)

print("")
print("Scraping termine ! Fichier cree : " + FICHIER_CSV)

## Etape 8 : Affichage des statistiques

On affiche les statistiques du dataset genere.

In [ ]:
afficher_statistiques(FICHIER_CSV)

## Etape 9 : Apercu du dataset

On affiche les 3 premieres lignes du fichier CSV pour verifier que tout est correct.

In [ ]:
import pandas as pd

# Lire le fichier CSV avec pandas
df = pd.read_csv(FICHIER_CSV)

# Afficher les 3 premieres lignes
print("Dimensions du dataset : " + str(df.shape[0]) + " lignes, " + str(df.shape[1]) + " colonnes")
print("")
df.head(3)

---

## Resultat final

Le fichier `articles.csv` est maintenant pret avec :
- **50 articles** au total (10 par categorie)
- **5 labels** : politique, sport, economie, sante, sciences
- **3 colonnes** : url, texte, label

Ce dataset sera utilise dans la **Partie 2** pour entrainer le modele Transformer CamemBERT.